In [1]:
import os

In [2]:
%pwd

'f:\\git-projects\\Text-Summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'f:\\git-projects\\Text-Summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            tokenizer_name = config.tokenizer_name
        )
        return data_transformation_config

In [8]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

f:\git-projects\Text-Summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here is the breakdown of why these specific attributes exist and what they represent.

### 1. How `input_encodings` has these attributes
When you call the tokenizer like this:
```python
input_encodings = self.tokenizer("Hello world", max_length=1024, truncation=True)
```
The `self.tokenizer` (which is an instance of a Hugging Face `AutoTokenizer`) returns a special object called a **`BatchEncoding`** (or just `Encoding` if it's a single string).

This object is essentially a **dictionary** that contains the raw numerical data required to feed into a neural network. It always contains specific keys:
*   `input_ids`: The numbers representing the words.
*   `attention_mask`: A binary list (0s and 1s).
*   `token_type_ids`: (Used for specific tasks like Next Sentence Prediction, usually ignored for summarization).

---

### 2. What is `input_ids`?
**`input_ids`** is the numerical representation of your text.

Neural networks cannot read text; they can only process numbers (matrices). The tokenizer converts every word or sub-word into a unique integer based on its vocabulary.

*   **Example:** If your vocabulary maps "Hello" to 100 and "world" to 200.
*   **Text:** `"Hello world"`
*   **`input_ids`:** `[100, 200]`

If you look at the code: `input_encodings['input_ids']` extracts that list of numbers `[100, 200]`.

---

### 3. What is `attention_mask` and why is it used?
`attention_mask` is used to handle **Padding**.

#### The Problem: Variable Lengths
Neural networks process data in **batches** (like Excel sheets or matrices). A fundamental rule of matrix math is that every row must have the same number of columns.
*   **Sentence A:** "I love cats" (3 words)
*   **Sentence B:** "I went to the store yesterday and bought some milk" (13 words)

If you want to train the model on both these sentences in one batch, you have a problem. The matrix needs to be uniform.

#### The Solution: Padding
You must pad the shorter sentence (Sentence A) with a special `[PAD]` token until it matches the length of the longest sentence in the batch (13 words).
*   **Sentence A becomes:** "I love cats [PAD] [PAD] ... [PAD]" (Total 13 words).

#### The Mask's Role
Now, the model sees the padded words. If the model treats `[PAD]` as real words, it will learn that "PAD" is a meaningful word, which will ruin the model's accuracy.

`attention_mask` tells the model **which parts of the sequence are real data and which are just filler.**
*   **1 (Real):** Keep looking at these words.
*   **0 (Padding):** Ignore these.

**Example:**
*   **Text:** `"I love cats"` (padded to length 5) -> `["I", "love", "cats", "[PAD]", "[PAD]"]`
*   **`input_ids`:** `[101, 23, 45, 102, 102]`
*   **`attention_mask`:** `[1, 1, 1, 0, 0]`

In the `convert_examples_to_features` function, `attention_mask` is passed into the model so the model knows not to waste compute on the zeros and not to count them as meaningful text.

In [15]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)
    
    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )
        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)
        
        return {
                'input_ids' : input_encodings['input_ids'],
                'attention_mask': input_encodings['attention_mask'],
                'labels': target_encodings['input_ids']
        }  
        
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features, 
            batched = True)
        
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))     
    
        

In [14]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-07-13 18:36:56,288: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-07-13 18:36:56,289: INFO: common]: yaml file: params.yaml loaded successfully
[2026-07-13 18:36:56,290: INFO: common]: created directory at: artifacts
[2026-07-13 18:36:56,290: INFO: common]: created directory at: artifacts/data_transformation
[2026-07-13 18:36:56,676: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-07-13 18:36:56,703: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-07-13 18:36:56,946: INFO: _client]: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-07-13 18:36:56,973: INFO: _client]: HTTP Request: HEAD https://huggingface.co/api/resolve-

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 188223.65 examples/s]
